# Check Transformer Block Counts From Weights

This notebook only inspects two model weight files and reports how many transformer blocks each one has.

Method: parse the weight keys that match `blocks.{layer_idx}.` and count unique `layer_idx` values.


In [ ]:
from pathlib import Path
import re
import torch

BLOCK_KEY_RE = re.compile(r"^blocks\.(\d+)\.")

def load_state_dict_from_weight(weight_path: str | Path):
    path = Path(weight_path)
    if not path.exists():
        raise FileNotFoundError(f"Weight file not found: {path}")

    payload = torch.load(path, map_location="cpu")

    if isinstance(payload, dict) and "model_state" in payload:
        state_dict = payload["model_state"]
        model_type = payload.get("model_type")
        model_config = payload.get("model_config", {})
    elif isinstance(payload, dict):
        state_dict = payload
        model_type = None
        model_config = {}
    else:
        raise TypeError(f"Unsupported weight payload type: {type(payload)!r}")

    if not isinstance(state_dict, dict):
        raise TypeError("Loaded model_state is not a dict-like state_dict.")

    return state_dict, {
        "path": str(path),
        "model_type": model_type,
        "config_n_layer": model_config.get("n_layer"),
    }

def count_transformer_blocks_from_state_dict(state_dict):
    layer_ids = set()
    example_keys = []

    for key in state_dict.keys():
        match = BLOCK_KEY_RE.match(key)
        if match is None:
            continue
        layer_idx = int(match.group(1))
        layer_ids.add(layer_idx)
        if len(example_keys) < 10:
            example_keys.append(key)

    ordered_layer_ids = sorted(layer_ids)
    return {
        "block_count": len(ordered_layer_ids),
        "layer_ids": ordered_layer_ids,
        "example_block_keys": example_keys,
    }

def inspect_weight(weight_path: str | Path):
    state_dict, meta = load_state_dict_from_weight(weight_path)
    block_info = count_transformer_blocks_from_state_dict(state_dict)
    return {**meta, **block_info}


In [ ]:
DEFAULT_DIR = Path("mini_gpt_attnres_runs/notebook_bootstrap")

MODEL_WEIGHTS = {
    "standard": DEFAULT_DIR / "ckpt_standard_last.pt",
    "attnres": DEFAULT_DIR / "ckpt_attnres_last.pt",  # edit if your attnres checkpoint is elsewhere
}

MODEL_WEIGHTS


In [ ]:
results = {}

for model_name, weight_path in MODEL_WEIGHTS.items():
    path = Path(weight_path)
    print(f"\n[{model_name}] {path}")

    if not path.exists():
        print("  status: missing file")
        print("  action: update MODEL_WEIGHTS and rerun this cell")
        continue

    info = inspect_weight(path)
    results[model_name] = info

    print(f"  model_type: {info['model_type']}")
    print(f"  config_n_layer: {info['config_n_layer']}")
    print(f"  block_count_from_weights: {info['block_count']}")
    print(f"  layer_ids: {info['layer_ids']}")

    if info['config_n_layer'] is not None and info['config_n_layer'] != info['block_count']:
        print("  warning: config_n_layer does not match counted block_count")

    print("  example_block_keys:")
    for key in info['example_block_keys']:
        print(f"    - {key}")

results
